In [1]:
import pandas as pd

# Morning: Jul 24th
## Case 1

In [ ]:
df = pd.read_parquet("../maganghub_data_master_2.parquet")

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Job ID               294 non-null    str  
 1   Keyword              294 non-null    str  
 2   Job Title            294 non-null    str  
 3   Company              294 non-null    str  
 4   Quota                294 non-null    int64
 5   Total Applicants     294 non-null    int64
 6   Approved Applicants  294 non-null    int64
 7   Education Levels     294 non-null    str  
 8   Working Days/Week    294 non-null    int64
 9   Location             294 non-null    str  
 10  Majors Allowed       294 non-null    str  
 11  Published At         294 non-null    str  
 12  Description          294 non-null    str  
dtypes: int64(4), str(9)
memory usage: 576.6 KB


In [14]:
df.head()

,Job ID,Keyword,Job Title,Company,Quota,Total Applicants,Approved Applicants,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description
0,a225414b-1dd6-4d80-8460-61b000b4d355,data analyst,Location Data Analyst Internship,PT Sumber Alfaria Trijaya Tbk,2,7,2,"Bachelor, Diploma",5,Kota Tangerang,"Teknik informatika, Sistem Informasi, Matematika",2026-07-16T09:57:10+07:00,"""1. Pemetaan & Analisa Potensi Pembukaan Toko ..."
1,a225a106-d1bc-4d87-9ce9-90ca9c68d5d3,data analytics,Data Analytics Visualization Support,Perusahaan Perseroan (Persero) PT. Asuransi Kr...,10,33,10,"Diploma, Bachelor, Profession",5,Kota Adm. Jakarta Pusat,"Aktuaria, Statistika, Sistem Informasi, Akunta...",2026-07-16T11:08:27+07:00,"Pengolahan & Pembersihan Data, Pembuatan Dashb..."
2,a22775e0-aa28-40c3-8d37-e815f72c0601,data analyst,Retail Data Analyst,Midea Electronics Indonesia,1,3,1,Bachelor,5,Kota Adm. Jakarta Pusat,"Manajemen, Teknik Industri, Sistem Informasi, ...",2026-07-16T10:56:36+07:00,"- Membantu mengumpulkan, memvalidasi, dan memp..."
3,a22fad39-91d6-4c3c-bd40-ebbdd2e8e66c,analisis data,Pengelolaan dan Analisis Data Operasional,PT Lotte Shopping Indonesia Store Tasikmalaya,2,24,2,Bachelor,5,Kota Tasikmalaya,"Teknik Informatika, Teknik Industri, Sistem In...",2026-07-16T10:13:01+07:00,"Mempelajari proses pengumpulan, pengolahan, an..."
4,a230d491-7cf9-46aa-8804-29acbfea1947,data analyst,Data Analyst (for FInance & Accounting),Jababeka Morotai,1,2,1,Bachelor,5,Kab. Bekasi,"Administrasi BIsnis, Manajemen, Akuntansi Keua...",2026-07-16T09:43:22+07:00,Analisa laporan keuangan dan rekonsiliasi data...


In [33]:
# 1. Calculate how many spots are actually left
df["Remaining Quota"] = df["Quota"] - df["Approved Applicants"]

# 2. Calculate how many people are still fighting for those spots
# We use .clip(lower=0) just in case Maganghub's data has a glitch
active_competitors = (df["Total Applicants"] - df["Approved Applicants"]).clip(lower=0)

# 3. Default everyone to 0 probability (this covers jobs that are already full)
df["Acceptance Probability"] = 0.0

# 4. Create a mask for jobs that actually still have open spots
open_spots = df["Remaining Quota"] > 0

# 5. Apply the real-world formula only to the open jobs
df.loc[open_spots, "Acceptance Probability"] = (
    df.loc[open_spots, "Remaining Quota"] / (active_competitors[open_spots] + 1)
)

# 6. Cap it at 1.0 (100%) for cases where Remaining Quota > Active Competitors
df["Acceptance Probability"] = df["Acceptance Probability"].clip(upper=1.0)

In [16]:
df.head()

,Job ID,Keyword,Job Title,Company,Quota,Total Applicants,Approved Applicants,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description,Remaining Quota,Acceptance Probability
0,a225414b-1dd6-4d80-8460-61b000b4d355,data analyst,Location Data Analyst Internship,PT Sumber Alfaria Trijaya Tbk,2,7,2,"Bachelor, Diploma",5,Kota Tangerang,"Teknik informatika, Sistem Informasi, Matematika",2026-07-16T09:57:10+07:00,"""1. Pemetaan & Analisa Potensi Pembukaan Toko ...",0,0.0
1,a225a106-d1bc-4d87-9ce9-90ca9c68d5d3,data analytics,Data Analytics Visualization Support,Perusahaan Perseroan (Persero) PT. Asuransi Kr...,10,33,10,"Diploma, Bachelor, Profession",5,Kota Adm. Jakarta Pusat,"Aktuaria, Statistika, Sistem Informasi, Akunta...",2026-07-16T11:08:27+07:00,"Pengolahan & Pembersihan Data, Pembuatan Dashb...",0,0.0
2,a22775e0-aa28-40c3-8d37-e815f72c0601,data analyst,Retail Data Analyst,Midea Electronics Indonesia,1,3,1,Bachelor,5,Kota Adm. Jakarta Pusat,"Manajemen, Teknik Industri, Sistem Informasi, ...",2026-07-16T10:56:36+07:00,"- Membantu mengumpulkan, memvalidasi, dan memp...",0,0.0
3,a22fad39-91d6-4c3c-bd40-ebbdd2e8e66c,analisis data,Pengelolaan dan Analisis Data Operasional,PT Lotte Shopping Indonesia Store Tasikmalaya,2,24,2,Bachelor,5,Kota Tasikmalaya,"Teknik Informatika, Teknik Industri, Sistem In...",2026-07-16T10:13:01+07:00,"Mempelajari proses pengumpulan, pengolahan, an...",0,0.0
4,a230d491-7cf9-46aa-8804-29acbfea1947,data analyst,Data Analyst (for FInance & Accounting),Jababeka Morotai,1,2,1,Bachelor,5,Kab. Bekasi,"Administrasi BIsnis, Manajemen, Akuntansi Keua...",2026-07-16T09:43:22+07:00,Analisa laporan keuangan dan rekonsiliasi data...,0,0.0


In [34]:
df[
    (df["Acceptance Probability"] != 0) &
    (
        df["Majors Allowed"].str.contains("matematika", case=False, na=False) | 
        df["Job Title"].str.contains("data scientist|data science", case=False, na=False)
    ) &
    (df["Location"].str.contains("jakarta|bandung", case=False, na=False))
][["Job Title", "Company", "Quota", "Total Applicants", "Approved Applicants", "Location", "Majors Allowed"]]

,Job Title,Company,Quota,Total Applicants,Approved Applicants,Location,Majors Allowed
124,Analis Data Tenaga Teknik Ketenagalistrikan,Direktorat Jenderal Ketenagalistrikan,4,12,2,Kota Adm. Jakarta Selatan,"Teknik Informatika, Fisika, Teknik Elektro, St..."
172,Data Analyst Intern,PT Jaminan Kredit Indonesia,3,2,1,Kota Adm. Jakarta Pusat,"Statistika, Matematika"
173,Network Business Data Analyst Intern,PT Jaminan Kredit Indonesia,2,1,1,Kota Adm. Jakarta Pusat,"Sains Data, Statistika, Matematika"
174,Business Data Analyst Intern,PT Jaminan Kredit Indonesia,2,1,1,Kota Adm. Jakarta Pusat,"Sains Data, Statistika, Matematika"


In [22]:
print(df.loc[129, "Description"])

Membantu tim dalam:1. Melakukan analisa data tenaga teknik ketenagalistrikan2. Melakukan analisa sistem informasi 3. Melakukan kegiatan administratif lainnya


In [ ]:
df[
    ((df["Acceptance Probability"] == 1) | (df["Acceptance Probability"] == 0.5)) &
    (df["Majors Allowed"].str.contains("matematika", case=False, na=False)) &
    (df["Location"].str.contains("jakarta", case=False, na=False))
][["Job Title", "Company", "Quota", "Total Applicants", "Approved Applicants", "Location", "Majors Allowed"]]

## Case 2 

In [35]:
df = pd.read_parquet("../maganghub_data_master_2.parquet")

# 1. Rename the misleading columns so your data is accurate
df = df.rename(columns={
    "Approved Applicants": "Real Quota", 
    "Quota": "Requested Quota (Ignore)"
})

# 2. Calculate the true probability
# If you apply, the pool becomes (Total Applicants + 1)
df["Acceptance Probability"] = df["Real Quota"] / (df["Total Applicants"] + 1)

# 3. Cap the probability at 1.0 (100%)
# This handles jobs where the Real Quota is larger than the number of applicants
df["Acceptance Probability"] = df["Acceptance Probability"].clip(upper=1.0)

# Preview your clean, mathematically sound dataframe
df[["Job Title", "Company", "Real Quota", "Total Applicants", "Acceptance Probability"]].head()

,Job Title,Company,Real Quota,Total Applicants,Acceptance Probability
0,Location Data Analyst Internship,PT Sumber Alfaria Trijaya Tbk,2,7,0.250000
1,Data Analytics Visualization Support,Perusahaan Perseroan (Persero) PT. Asuransi Kr...,10,33,0.294118
2,Retail Data Analyst,Midea Electronics Indonesia,1,3,0.250000
3,Pengelolaan dan Analisis Data Operasional,PT Lotte Shopping Indonesia Store Tasikmalaya,2,24,0.080000
4,Data Analyst (for FInance & Accounting),Jababeka Morotai,1,2,0.333333


In [41]:
df[
    # (df["Acceptance Probability"] > 0.5) &
    (
        df["Majors Allowed"].str.contains("matematika", case=False, na=False) | 
        df["Job Title"].str.contains("data scientist|data science", case=False, na=False)
    ) &
    (df["Location"].str.contains("jakarta|bandung", case=False, na=False)) &
    (df["Real Quota"] > df["Total Applicants"])
][["Job Title", "Company", "Real Quota", "Total Applicants","Location", "Majors Allowed", "Acceptance Probability"]]

,Job Title,Company,Real Quota,Total Applicants,Location,Majors Allowed,Acceptance Probability
83,Regional Data Analyst Intern,PT Sinarniaga Sejahtera,1,0,Kota Adm. Jakarta Timur,"Teknik lndustri, Manajemen, sistem informasi, ...",1.0
181,Data Science,PT. Asuransi Etiqa Internasional Indonesia,1,0,Kota Adm. Jakarta Pusat,Ilmu Komputer,1.0


In [48]:
df[
    # (df["Acceptance Probability"] > 0.5) &
    (
        df["Majors Allowed"].str.contains("matematika|statistika|statistik|data", case=False, na=False) | 
        df["Job Title"].str.contains("data scientist|data science", case=False, na=False)
    ) &
    (df["Location"].str.contains("jakarta", case=False, na=False)) &
    (df["Real Quota"] > df["Total Applicants"])
][["Job Title", "Company", "Real Quota", "Total Applicants","Location", "Majors Allowed", "Acceptance Probability"]]

,Job Title,Company,Real Quota,Total Applicants,Location,Majors Allowed,Acceptance Probability
83,Regional Data Analyst Intern,PT Sinarniaga Sejahtera,1,0,Kota Adm. Jakarta Timur,"Teknik lndustri, Manajemen, sistem informasi, ...",1.0
119,Analis Data Penyediaan Listrik Kemasyarakatan,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,"Teknik Informatika, Ilmu Komputer, Sistem Info...",1.0
181,Data Science,PT. Asuransi Etiqa Internasional Indonesia,1,0,Kota Adm. Jakarta Pusat,Ilmu Komputer,1.0
214,DPPL - Asisten Analis Data Statistik,OJKI,1,0,Kota Adm. Jakarta Selatan,Statistik,1.0
266,Analis data pengolahan hasil Kelautan dan Peri...,Direktorat Pengolahan,1,0,Kota Adm. Jakarta Pusat,Statistik,1.0
280,Analis Data Kebijakan/Analis Monitoring dan Ev...,Kementerian Pendayagunaan Aparatur Negara dan ...,1,0,Kota Adm. Jakarta Selatan,Statistika,1.0
288,Analis Data Pengawasan Pengelolaan Pengaduan N...,Ombudsman Republik Indonesia,1,0,Kota Adm. Jakarta Selatan,"Teknik Informatika, Ilmu Komputer, Statistika ...",1.0


In [46]:
df[
    # (df["Acceptance Probability"] > 0.5) &
    # (
    #     df["Majors Allowed"].str.contains("matematika|statistika|data", case=False, na=False) | 
    #     df["Job Title"].str.contains("data scientist|data science", case=False, na=False)
    # ) &
    (df["Location"].str.contains("jakarta pusat|jakarta selatan", case=False, na=False)) &
    (df["Real Quota"] > df["Total Applicants"])
][["Job Title", "Company", "Real Quota", "Total Applicants","Location", "Majors Allowed", "Acceptance Probability"]]

,Job Title,Company,Real Quota,Total Applicants,Location,Majors Allowed,Acceptance Probability
77,Analis Data dan Informasi,Rumah Sakit Umum Dr Cipto Mangun Kusumo Jakart...,1,0,Kota Adm. Jakarta Pusat,Manajemen Sumber Daya Manusia,1.0
80,Analisis Data dan Informasi,Rumah Sakit Umum Dr Cipto Mangun Kusumo Jakart...,1,0,Kota Adm. Jakarta Pusat,Kedokteran,1.0
92,Analis Data Perencananaan Kebijakan Tenaga Lis...,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,Teknik Elektro,1.0
102,Analis Data Pengaturan Operasi Usaha Ketenagal...,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,"Teknik Mesin, Teknik Elektro",1.0
106,Analis Data Harga Tenaga Listrik,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,Teknik Elektro,1.0
119,Analis Data Penyediaan Listrik Kemasyarakatan,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,"Teknik Informatika, Ilmu Komputer, Sistem Info...",1.0
127,Analis Data Usaha Jasa Penunjang Ketenagalistr...,Direktorat Jenderal Ketenagalistrikan,1,0,Kota Adm. Jakarta Selatan,"Kearsipan, Manajemen Rekod dan Arsip",1.0
155,Systems and Data Analyst,Sekretariat Direktorat Jenderal Prasarana dan ...,1,0,Kota Adm. Jakarta Selatan,"Teknologi Informasi, Sistem dan Teknologi Info...",1.0
181,Data Science,PT. Asuransi Etiqa Internasional Indonesia,1,0,Kota Adm. Jakarta Pusat,Ilmu Komputer,1.0
214,DPPL - Asisten Analis Data Statistik,OJKI,1,0,Kota Adm. Jakarta Selatan,Statistik,1.0


# Afternoon: Jul 24th

In [2]:
df = pd.read_parquet("../maganghub_data_master_3.parquet")

# 2. Calculate the true probability
# If you apply, the pool becomes (Total Applicants + 1)
df["Acceptance Probability"] = df["Real Quota"] / (df["Total Applicants"] + 1)

# 3. Cap the probability at 1.0 (100%)
# This handles jobs where the Real Quota is larger than the number of applicants
df["Acceptance Probability"] = df["Acceptance Probability"].clip(upper=1.0)

# Preview your clean, mathematically sound dataframe
df[["Job Title", "Company", "Real Quota", "Total Applicants", "Acceptance Probability"]].head()

,Job Title,Company,Real Quota,Total Applicants,Acceptance Probability
0,Location Data Analyst Internship,PT Sumber Alfaria Trijaya Tbk,2,7,0.250000
1,Data Analytics Visualization Support,Perusahaan Perseroan (Persero) PT. Asuransi Kr...,10,33,0.294118
2,Retail Data Analyst,Midea Electronics Indonesia,1,3,0.250000
3,Pengelolaan dan Analisis Data Operasional,PT Lotte Shopping Indonesia Store Tasikmalaya,2,22,0.086957
4,Data Analyst (for FInance & Accounting),Jababeka Morotai,1,3,0.250000


In [16]:
df[
    (
        df["Majors Allowed"].str.contains("matematika|statistika|statistik|data", case=False, na=False) | 
        df["Job Title"].str.contains("data scientist|data science", case=False, na=False)
    ) &
    (df["Location"].str.contains("jakarta", case=False, na=False)) &
    (df["Real Quota"] > df["Total Applicants"])
][["Job Title", "Company", "Real Quota", "Total Applicants","Location", "Majors Allowed", "Acceptance Probability"]]

,Job Title,Company,Real Quota,Total Applicants,Location,Majors Allowed,Acceptance Probability


In [3]:
df[df["Company"] == "PT. Asuransi Etiqa Internasional Indonesia"]

,Job ID,Keyword,Job Title,Company,Requested Quota (Ignored),Total Applicants,Real Quota,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description,Acceptance Probability
187,a24335df-fa6e-4ff2-b3e3-91a0784e2657,data science,Data Science,PT. Asuransi Etiqa Internasional Indonesia,1,2,1,Bachelor,5,Kota Adm. Jakarta Pusat,Ilmu Komputer,2026-07-16T09:52:41+07:00,Kami sedang mencari Data Science Intern yang b...,0.333333


In [4]:
df[
    (df["Job Title"] == "Analis Data dan Informasi") &
    (df["Company"] == "Kementerian Usaha Mikro, Kecil, dan Menengah")
]

,Job ID,Keyword,Job Title,Company,Requested Quota (Ignored),Total Applicants,Real Quota,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description,Acceptance Probability
289,a243fe13-71bd-4fbe-9fa8-114f9d5a0fa1,analis data,Analis Data dan Informasi,"Kementerian Usaha Mikro, Kecil, dan Menengah",1,2,1,Bachelor,5,Kota Adm. Jakarta Selatan,Ilmu Komputer,2026-07-16T12:04:54+07:00,1. Melakukan analisis dan pengelolaan data\n2....,0.333333


In [5]:
df[
    # (
        # df["Majors Allowed"].str.contains("matematika|statistika|statistik|data", case=False, na=False) | 
        # df["Job Title"].str.contains("data scientist|data science", case=False, na=False) 
        # df["Description"].str.contains("matematika|mathematics|math", case=False, na=False)
    # ) &
    (df["Location"].str.contains("jakarta pusat|jakarta selatan", case=False, na=False)) &
    (df["Real Quota"] > df["Total Applicants"])
][["Job Title", "Company", "Real Quota", "Total Applicants","Location", "Majors Allowed", "Acceptance Probability"]]

,Job Title,Company,Real Quota,Total Applicants,Location,Majors Allowed,Acceptance Probability


In [ ]:
# https://maganghub.kemnaker.go.id/magang-nasional/lowongan/analis-data-dan-informasi-a243fe13-71bd-4fbe-9fa8-114f9d5a0fa1

In [21]:
df[df["Description"].str.contains("matematika|mathematics|math", case=False, na=False)]

,Job ID,Keyword,Job Title,Company,Requested Quota (Ignored),Total Applicants,Real Quota,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description,Acceptance Probability
34,a23ddb74-a2b0-4734-95fe-e89dfe379d02,analis data,Analis Data dan Informasi,Balai Pemberdayaan Industri Persepatuan Indonesia,3,9,3,Bachelor,5,Kab. Sidoarjo,"Sains Data, Statistika dan Sains Data, Sains D...",2026-07-16T13:08:16+07:00,Mengumpulkan data dari berbagai unit layanan m...,0.300000
64,a23f7715-2db0-4ab9-b24f-3358846acc43,analis data,Analis Data dan Informasi Kesehatan,Rumah Sakit Umum Daerah Bhakti Dharma Husada K...,1,6,1,"Diploma, Bachelor",5,Kota Surabaya,Statistika dan Sains Data,2026-07-16T11:12:16+07:00,1. Mengolah data mentah menjadi informasi yang...,0.142857
86,a23fe32a-a70f-4c89-aa5e-bac616157192,data analyst,Data Analyst Supporting,Perusahaan Perseroan (Persero) PT. Len Industri,1,13,1,"Diploma, Bachelor",5,Kota Bandung,"Teknik Informatika, Sains Data, Statistik, Sis...",2026-07-16T10:12:07+07:00,Kualifikasi:\n- Fresh graduate jurusan Data Sc...,0.071429
90,a24002f5-fe86-4d33-8a28-2df63ff74b16,data sains,Data Sains Supporting,Perusahaan Perseroan (Persero) PT. Len Industri,1,9,1,"Bachelor, Diploma",5,Kota Bandung,"Ilmu Komputer, Statistik, Sistem Informasi, Ad...",2026-07-16T10:12:07+07:00,Kualifikasi:\n- Fresh graduate jurusan Data Sc...,0.100000
173,a241bfc3-875e-4ea4-b900-ea5ddcd75b89,data science,Data Science Intern,"PT. Blue Bird, Tbk",1,7,1,Bachelor,5,Kota Adm. Jakarta Selatan,"Sains Data, Ilmu Komputer, Sistem Informasi, S...",2026-07-16T10:17:42+07:00,"Job Deskripsi:\n• Membantu mengolah, membersih...",0.125000
174,a241c088-1e7f-445e-a22d-859aa271fbfa,data analytics,Data Analytics Intern,"PT. Blue Bird, Tbk",1,6,1,Bachelor,5,Kota Adm. Jakarta Selatan,"Sains Data, Ilmu Komputer, Ekonomi, Statistika...",2026-07-16T10:17:42+07:00,"Job Deskripsi:• Mendukung pembuatan dashboard,...",0.142857
241,a243abe2-ff3c-4a9f-94f4-5f569d43946a,data analis,Data Analis,Pusat Pelatihan Kelautan dan Perikanan,1,2,1,Bachelor,5,Kota Adm. Jakarta Pusat,"Statistika, Manajemen Informatika, Matematika",2026-07-16T12:30:21+07:00,Data Analis adalah jabatan yang bertugas mengu...,0.333333
285,a243f6bf-960f-480d-a8f7-4e2dec3c0c8d,analis data,Analis Data Komunikasi,Biro Hubungan Masyarakat dan Kerja Sama Luar N...,2,12,2,"Diploma, Bachelor",5,Kota Adm. Jakarta Pusat,"Jurnalistik, Ilmu Hubungan Internasional, Ilmu...",2026-07-16T13:04:54+07:00,Kualifikasi:1. Fresh graduate S1 dari bidang I...,0.153846


In [6]:
df[df["Company"].str.contains("pertamina", case=False, na=False)]

,Job ID,Keyword,Job Title,Company,Requested Quota (Ignored),Total Applicants,Real Quota,Education Levels,Working Days/Week,Location,Majors Allowed,Published At,Description,Acceptance Probability
